In [3]:
!pip install -q \
    huggingface-hub==1.23.0 \
    transformers \
    datasets==3.5.0 \
    accelerate==1.6.0 \
    peft==0.15.0 \
    bitsandbytes==0.49.0

In [4]:
!pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 6.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 9.8 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]


In [5]:
import torch

# 메모리 사용량
def print_gpu_utilization() :
  if torch.cuda.is_available():
    used_memory = torch.cuda.memory_allocated()
    print(f"GPU메모리 사용량 = {used_memory}")
  else :
    print("런타임 cpu")

print_gpu_utilization()

런타임 cpu


In [ ]:
# 모델의 메모리 사용량
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_and_tokenizer(model_id, peft=None) :
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if peft is None :
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map = {"":0})

        print_gpu_utilization()
        return model, tokenizer

model_id = 'EleutherAI/polyglot-ko-1.3b'
model, tokenizer = load_model_and_tokenizer(model_id)
print("모델 파라미터 데이터 타입: ", model.dtype)

/Users/jeon-eunkyu/miniforge3/envs/seoulstationv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files:   0%|          | 0/3 [00:34<?, ?it/s]


In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

def estimate_memory_of_gradients(model) :
    total_memory = 0

    for param in model.parameters():
        if param.grad is not None:
            total_memory += param.grad.nelement() * param.grad.element_size() # gradient 값의 수 * 데이터 크기
    return total_memory

def estimate_memory_of_optimizer(optimizer) :
    total_memory = 0
    
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                total_memory += v.element() * v.elememt_size() # optimizer에 저장된 값의 수 * 데이터 크기
    return total_memory


In [ ]:
from torch.optim import optimizer
from transformers.models.kyutai_speech_to_text import processing_kyutai_speech_to_text
def train_model(model, dataset, training_args):
    if training_args.gradient_checkpointing:
        model.gradient_checkpointing_enable()

    train_dataloader = DataLoader(dataset, batch_size = training_args.per_device_train_batch_size)
    optimizer = AdamW(model.parameters())
    model.train()

    gpu_utilization_printed = False

    for step, batch in enumerate(train_dataloader, start = 1):
        batch = {k: v.to(model.device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        loss = loss / training_args.gradient_accumulation_steps
        loss.backward()

        if step % training_args.gradient_accumulation_steps == 0:
            optimizer.step()
            gradients_memory = estimate_memory_of_gradients(model)
            optimizer_memory = estimate_memory_of_optimizer(optimizer)

            if not gpu_utilization_printed:
                print_gpu_utilization()
                gpu_utilization_printed = True

            optimizer.zero_grad()
        print(f"옵티마이저 상태의 메모리 : {optimizer_memory / (1024 ** 3):.3f}GB")
        print(f"그래디언트 상태의 메모리 : {gradients_memory / (1024 ** 3):.3f}GB")

In [ ]:
import numpy as np
from datasets import Dataset

